# Import libraries

In [1]:
import pandas as pd
import numpy as np

# Load dataset

In [2]:
df = pd.read_csv(r"C:\Users\iroge\OneDrive\Desktop\OIBSIP\dirty_cafe_sales.csv")

# Basic inspection

In [3]:
print("Dataset shape:", df.shape)

Dataset shape: (10000, 8)


In [4]:
display(df.head())
display(df.tail())


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
9995,TXN_7672686,Coffee,2,2.0,4.0,NaN,UNKNOWN,2023-08-30
9996,TXN_9659401,NaN,3,NaN,3.0,Digital Wallet,NaN,2023-06-02
9997,TXN_5255387,Coffee,4,2.0,8.0,Digital Wallet,NaN,2023-03-02
9998,TXN_7695629,Cookie,3,NaN,3.0,Digital Wallet,NaN,2023-12-02
9999,TXN_6170729,Sandwich,3,4.0,12.0,Cash,In-store,2023-11-07


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


# Missing values per column

In [6]:
missing_values = df.isnull().sum()

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing_values,
    "Missing Percentage": missing_percentage.round(2)
})

display(missing_report)

,Missing Count,Missing Percentage
Transaction ID,0,0.00
Item,333,3.33
Quantity,138,1.38
Price Per Unit,179,1.79
Total Spent,173,1.73
Payment Method,2579,25.79
Location,3265,32.65
Transaction Date,159,1.59


# Count duplicate rows

In [7]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


# Number of unique values in each column

In [8]:
unique_report = pd.DataFrame({
    "Unique Values": df.nunique(),
    "Sample Values": [
        df[col].dropna().unique()[:10]
        for col in df.columns
    ]
})

display(unique_report)

,Unique Values,Sample Values
Transaction ID,10000,"[TXN_1961373, TXN_4977031, TXN_4271903, TXN_70..."
Item,10,"[Coffee, Cake, Cookie, Salad, Smoothie, UNKNOW..."
Quantity,7,"[2, 4, 5, 3, 1, ERROR, UNKNOWN]"
Price Per Unit,8,"[2.0, 3.0, 1.0, 5.0, 4.0, 1.5, ERROR, UNKNOWN]"
Total Spent,19,"[4.0, 12.0, ERROR, 10.0, 20.0, 9.0, 16.0, 15.0..."
Payment Method,5,"[Credit Card, Cash, UNKNOWN, Digital Wallet, E..."
Location,4,"[Takeaway, In-store, UNKNOWN, ERROR]"
Transaction Date,367,"[2023-09-08, 2023-05-16, 2023-07-19, 2023-04-2..."


# Save baseline information before cleaning

In [20]:

before_rows = len(df)
before_duplicates = df.duplicated().sum()
before_nulls = df.isnull().sum().sum()

print("Rows:", before_rows)
print("Duplicate rows:", before_duplicates)
print("Total missing values:", before_nulls)

Rows: 10000
Duplicate rows: 0
Total missing values: 6826


# Quality report

In [21]:
quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Null Count": df.isnull().sum().values,
    "Null %": (df.isnull().mean() * 100).round(2).values,
    "Unique Values": df.nunique().values
})

display(quality_report)

,Column,Data Type,Null Count,Null %,Unique Values
0,Transaction ID,object,0,0.00,10000
1,Item,object,333,3.33,10
2,Quantity,object,138,1.38,7
3,Price Per Unit,object,179,1.79,8
4,Total Spent,object,173,1.73,19
5,Payment Method,object,2579,25.79,5
6,Location,object,3265,32.65,4
7,Transaction Date,object,159,1.59,367


# Quality report

In [22]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


# Range anomalies

In [23]:
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].describe())

          Quantity  Price Per Unit  Total Spent
count  9521.000000     9467.000000  9498.000000
mean      3.028463        2.949984     8.924352
std       1.419007        1.278450     6.009919
min       1.000000        1.000000     1.000000
25%       2.000000        2.000000     4.000000
50%       3.000000        3.000000     8.000000
75%       4.000000        4.000000    12.000000
max       5.000000        5.000000    25.000000


## Missing Data Handling Strategy

The dataset contains both genuine missing values and invalid text values such as
`UNKNOWN` and `ERROR`.

For numeric columns (`Quantity`, `Price Per Unit`, and `Total Spent`), invalid
values are converted to missing values. Median imputation is used because the
median is less sensitive to extreme values than the mean.

For categorical columns (`Item`, `Payment Method`, and `Location`), mode
imputation is used because these variables represent categories rather than
continuous numerical measurements.

For `Transaction Date`, invalid and missing dates are retained as missing
rather than forward-filled because transaction dates should not be inferred
from another transaction.

Rows are not deleted solely because they contain missing values, allowing us
to preserve as much transaction information as possible.

# Replace invalid text values with NaN

In [24]:

df = df.replace(["UNKNOWN", "ERROR"], np.nan)

In [25]:
for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    df[col] = df[col].fillna(df[col].median())

In [26]:
for col in ["Item", "Payment Method", "Location"]:
    df[col] = df[col].fillna(df[col].mode()[0])

In [27]:
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [28]:
df = df.dropna(subset=["Transaction Date"])

## Duplicate Removal

Duplicate transactions can distort analysis and cause transactions to be
counted more than once. Exact duplicate rows are therefore identified and
removed. The number of removed records is documented below.

In [29]:
duplicates_before = df.duplicated().sum()

df = df.drop_duplicates()

duplicates_after = df.duplicated().sum()

print("Duplicates before removal:", duplicates_before)
print("Duplicates after removal:", duplicates_after)
print("Duplicates removed:", duplicates_before - duplicates_after)


Duplicates before removal: 0
Duplicates after removal: 0
Duplicates removed: 0


## Standardisation

Text-based categorical fields are standardised by removing unnecessary
whitespace and applying consistent capitalization. Transaction dates are
converted to pandas datetime format.

# Standardise text columns

In [30]:

text_cols = ["Item", "Payment Method", "Location"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

In [31]:
df["Payment Method"] = df["Payment Method"].str.title()
df["Location"] = df["Location"].str.title()

## Outlier Detection

The Interquartile Range (IQR) method is used to identify statistical
outliers in the numeric columns. The IQR is calculated as Q3 − Q1, and
values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR are classified as
potential outliers.

Outliers are investigated rather than automatically deleted. Since extreme
values may represent legitimate transactions, they are retained unless they
are determined to be data-entry errors.

# Detect outliers using IQR

In [32]:


outlier_report = []

for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ]
    
    outlier_report.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": len(outliers)
    })

outlier_report = pd.DataFrame(outlier_report)

display(outlier_report)

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,Quantity,2.0,4.0,2.0,-1.0,7.0,0
1,Price Per Unit,2.0,4.0,2.0,-1.0,7.0,0
2,Total Spent,4.0,12.0,8.0,-8.0,24.0,250


In [33]:
# Correct data types

In [34]:


df["Transaction ID"] = df["Transaction ID"].astype("string")

df["Item"] = df["Item"].astype("string")

df["Quantity"] = df["Quantity"].round().astype("Int64")

df["Price Per Unit"] = df["Price Per Unit"].astype(float)

df["Total Spent"] = df["Total Spent"].astype(float)

df["Payment Method"] = df["Payment Method"].astype("string")

df["Location"] = df["Location"].astype("string")

df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9540 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    9540 non-null   string        
 1   Item              9540 non-null   string        
 2   Quantity          9540 non-null   Int64         
 3   Price Per Unit    9540 non-null   float64       
 4   Total Spent       9540 non-null   float64       
 5   Payment Method    9540 non-null   string        
 6   Location          9540 non-null   string        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: Int64(1), datetime64[ns](1), float64(2), string(4)
memory usage: 680.1 KB


# Final Validation

In [36]:
print("Final shape:", df.shape)
print("\nFinal missing values:")
display(df.isnull().sum())

print("\nFinal data types:")
display(df.dtypes)

print("\nFinal duplicate count:", df.duplicated().sum())

Final shape: (9540, 8)

Final missing values:


Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


Final data types:


Transaction ID      string[python]
Item                string[python]
Quantity                     Int64
Price Per Unit             float64
Total Spent                float64
Payment Method      string[python]
Location            string[python]
Transaction Date    datetime64[ns]
dtype: object


Final duplicate count: 0


# Save the Clean Dataset

In [37]:
df.to_csv("cleaned_cafe_sales.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


# Map down

dirty_cafe_sales.csv
        ↓
Data Quality Report
        ↓
Missing Data Handling
        ↓
Duplicate Removal
        ↓
Standardisation
        ↓
Outlier Detection
        ↓
Data Type Correction
        ↓
Before vs After Report
        ↓
cleaned_cafe_sales.csv